# Differentiable Monte Carlo — The Real Deal

**Goal:** make the MC PLE pipeline differentiable so we can recover parameters by
gradient descent instead of grid search.

This notebook makes the **line width `gamma`** differentiable end-to-end:
`loss = mmd2_biased(log10(sim), log10(real)); loss.backward()` populates
`gamma.grad`. Two pieces (per the journal): reparameterized sampling so `gamma`
flows into the photons, and **implicit differentiation** through the per-run fit
(no unrolling of the optimizer).

`nbar` (mean photon count) is held fixed this round — its discrete count needs a
finite-difference surrogate, which is the next step.

---

## Step 1: Imports & Setup

In [1]:
import math, time
import numpy as np
import torch
import torch.func as tfunc
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

torch.set_default_dtype(torch.float32)

# Model definitions from src/
from src.fitting import (
    FREQ_MIN, FREQ_MAX, UNIFORM_DENSITY, WIDTH_MAX, WIDTH_EPS, FIT_ETA,
    _width, _raw_from_width, _lorentz_cdf, _normal_cdf,
    log_pdf, nll, fwhm_from_theta, fit_profile,
    run_one_ple_scan, voigt_fwhm
)
from src.samplers import signal_detunings, build_photons, draw_fixed_noise
from src.losses import wasserstein_loss, per_run_wasserstein_sorted

print('All imports from src/')


from dataclasses import dataclass


@dataclass
class MCParams:
    nbar: float
    sigma: float = 6.0
    lambda_: float = 2.0


@dataclass
class SimParams:
    gamma: float = 20.0
    sigma_n: float = 6.0
    lambda_: float = 2.0


All imports from src/


## Step 2: Parameters & Reparameterized Sampling

Each run draws a photon count `N`, then `N` signal detunings from a Lorentzian
(Cauchy) line of HWHM `gamma`, plus a uniform background. Two changes from the
original forward-only code make this differentiable and correct:

1. **Reparameterization** — the randomness is fixed per run as quantiles `u` (and
   background `b`, count `N`); the signal detunings `gamma * tan(pi*(u-0.5))` are
   then a *differentiable function of gamma*.
2. **Truncation, not clipping** — photons outside `[FREQ_MIN, FREQ_MAX]` are *not
   detected* (dropped), rather than piled onto the window edges. Clipping-to-edge
   created spurious spikes that biased the fit and broke monotonicity of FWHM in
   `gamma`; truncation (with the truncation-aware likelihood in Step 3) is both
   physically correct and recovers `gamma` cleanly.

In [2]:
# signal_detunings / build_photons imported from src/samplers.py
print('Samplers from src/')


Samplers from src/


## Step 3: Fitting — Truncation-aware pseudo-Voigt + Background (MLE)

To mirror the experiment we fit a **Voigt** profile, not a pure Lorentzian. The
paper samples a Cauchy line but, *following the experimental procedure*, fits each
PLE scan with a **Voigt** (Gaussian ⊗ Lorentzian). Matching that estimator is what
makes the simulated FWHM distribution comparable to the real one — the Voigt fit's
characteristic low-photon bias is precisely the effect the Monte Carlo method
reproduces. (Sampling pure-Cauchy but fitting Voigt is deliberate: true physical
line shape in, experimental estimator out, on both sides.)

We use a **pseudo-Voigt** — a convex mix `eta·Gaussian + (1-eta)·Lorentzian` sharing
one center — because it is analytically differentiable and so plugs straight into
the implicit-diff machinery (Step 4). We keep the **uniform background** and
**renormalize each signal component over the detection window** so the likelihood
is correct for truncated data.

**`eta` is fixed for now** (`FIT_ETA = 0.2`). Freeing it made the fit degenerate —
`eta` saturated, `gamma` diverged, and the per-run implicit-diff gradient became
unreliable (median error ~38% vs ~0.4% with `eta` fixed). A small fixed value is
also physically apt: the paper's NV line is *predominantly power broadened* (mostly
Lorentzian). It is a single constant to free up again later.

Remaining params (unconstrained for the optimizer): `theta = (center, raw_gamma,
raw_sigma, logit_w)`. The two widths are **bounded to the detection window** —
`gamma, sigma_g = WIDTH_MAX·sigmoid(raw)` — since a width wider than the recorded
window is meaningless and, left unbounded, `gamma` diverges. Signal weight
`w = sigmoid(logit_w)`. The reported line width is the Voigt **FWHM** combining
both component widths (Olivero–Longbothum).

In [3]:
# Model functions imported from src/fitting.py above.
# This cell kept as a reminder that everything is now in src/.
print('Fitting functions from src/')


Fitting functions from src/


## Step 4: Differentiating Through the Fit — Implicit Function Theorem

We want `dFWHM/dgamma` for one run, but the FWHM comes out of a fit — an `argmin`
of the NLL over `theta` — so there's no cheap way to backprop through the
optimizer's iterations. The trick: we never differentiate the *iterations*, only
the *condition that defines the optimum*.

At the optimum `theta*`, the fit has converged, so the NLL's slope in `theta` is
zero:

$$\frac{dNLL}{d\theta}\Big|_{\theta^*} = 0.$$

Now nudge `gamma`. `gamma` enters the NLL **only through the photon positions**
`build_photons(gamma, u, b)` — same frozen `u`, no photon dropped — so moving
`gamma` shifts the data, which moves the optimum `theta*`. But that zero-slope
condition has to keep holding at the new optimum. Asking "how must `theta*` move to
keep the slope at zero?" gives us `dtheta*/dgamma` in terms of two second
derivatives of the NLL (both computed by `torch.func` in `hessian_blocks`):

- **`H_tt`** — how the theta-slope `dNLL/dtheta` changes as `theta` changes
  (curvature of the NLL in the fit params; a 4×4).
- **`H_tg`** — how that same theta-slope changes as `gamma` changes, i.e. through
  the shifting data (a 4×1). This is the only place `gamma` couples in.

Keeping the slope pinned at zero means the `theta` change and the `gamma` change
must cancel, which rearranges to

$$\frac{d\theta^*}{d\gamma} = -\,H_{tt}^{-1}\,H_{tg}$$

(`dfwhm_dgamma` solves this linear system rather than inverting explicitly).
Finally the reported width is `fwhm_from_theta(theta*)`, so we chain-rule through
it — `dFWHM/dtheta` from autograd — to get the scalar we actually want:

$$\frac{dFWHM}{d\gamma} = \frac{dFWHM}{d\theta}\cdot\frac{d\theta^*}{d\gamma}.$$

We wrap one run as a `torch.autograd.Function`: forward runs the (detached) fit,
backward returns `dFWHM/dgamma` via the formula above, so it composes with the
outer MMD² graph automatically. Ill-conditioned runs (rare, bad fits) return a
zero gradient instead of a garbage one.

In [4]:
N_PARAMS = 4  # theta = (center, raw_gamma, raw_sigma, logit_w); eta is fixed (FIT_ETA)


def hessian_blocks(theta_star, gamma, u, b):
    """Return (H_tt 4x4, H_tg 4x1) of the inner NLL at (theta*, gamma).

    gamma enters the NLL only through photons(gamma) (same frozen `u`, no photon
    dropped). Uses torch.func: H_tt = Jac_theta(grad_theta), H_tg = Jac_gamma(grad_theta).
    """
    def loss_of(theta, g):
        return nll(theta, build_photons(g, u, b))

    grad_theta = tfunc.grad(loss_of, argnums=0)
    H_tt = tfunc.jacrev(grad_theta, argnums=0)(theta_star, gamma)
    H_tg = tfunc.jacrev(grad_theta, argnums=1)(theta_star, gamma).reshape(N_PARAMS, 1)
    return H_tt, H_tg


def dfwhm_dgamma(theta_star, gamma, u, b):
    """Implicit-diff scalar dFWHM/dgamma at the optimum; None if ill-conditioned.

    Implicit function theorem: dtheta*/dgamma = -H_tt^{-1} H_tg, then chain-rule
    through the Voigt width: dFWHM/dgamma = (dFWHM/dtheta) . dtheta*/dgamma, with
    dFWHM/dtheta obtained by autograd of `fwhm_from_theta`.

    Scale-aware Tikhonov regularization on H_tt; a non-PD or badly conditioned
    Hessian (rare, on degenerate fits) or any LinAlg failure returns None so the caller zeroes that
    run's contribution. The per-run gradient is clamped to a sane range.
    """
    H_tt, H_tg = hessian_blocks(theta_star, gamma, u, b)
    if not (torch.isfinite(H_tt).all() and torch.isfinite(H_tg).all()):
        return None
    reg = 1e-6 * (H_tt.diagonal().abs().mean() + 1e-12)
    H = H_tt + reg * torch.eye(N_PARAMS, dtype=H_tt.dtype)
    try:
        eig = torch.linalg.eigvalsh(H)
        if eig.min() <= 0 or (eig.max() / eig.min()) > 1e10:
            return None
        dtheta_dg = -torch.linalg.solve(H, H_tg)
    except torch._C._LinAlgError:
        return None
    v = tfunc.grad(fwhm_from_theta)(theta_star).to(H_tt.dtype)  # dFWHM/dtheta, shape (4,)
    return torch.clamp((v @ dtheta_dg).squeeze(), -10.0, 10.0)


class DifferentiableRun(torch.autograd.Function):
    """One MC run as an autograd node: gamma -> FWHM, with implicit-diff backward."""

    @staticmethod
    def forward(ctx, gamma, u, b):
        g = gamma.detach()
        photons = build_photons(g, u, b)
        theta_star = fit_profile(photons)
        if theta_star is None:
            ctx.failed = True
            ctx.save_for_backward(g)
            return 2.0 * g  # differentiable proxy fallback (rare)
        ctx.failed = False
        ctx.save_for_backward(g, u, b, theta_star)
        return fwhm_from_theta(theta_star)

    @staticmethod
    def backward(ctx, grad_out):
        if ctx.failed:
            return grad_out * 2.0, None, None  # d(2*gamma)/dgamma = 2
        g, u, b, theta_star = ctx.saved_tensors
        grad = dfwhm_dgamma(theta_star, g, u, b)
        if grad is None:
            return grad_out * 0.0, None, None
        return grad_out * grad, None, None


def run_fwhm(gamma, u, b):
    """Forward-only FWHM for a single run (used for finite-difference checks)."""
    photons = build_photons(gamma, u, b)
    theta_star = fit_profile(photons)
    if theta_star is None:
        return 2.0 * float(gamma)
    return float(fwhm_from_theta(theta_star))

## Step 5: Simulation

`simulate_diff` runs many MC runs over a fixed noise set and stacks the FWHMs into
a tensor that is **differentiable in gamma**. `simulate_target` is the forward-only
version used to make the (detached) "real" data we fit against.

In [5]:
def draw_noise_set(n_runs, params, rng):
    """A list of per-run fixed noise tuples (u, b, N) for one simulation."""
    return [draw_fixed_noise(params.nbar, params.sigma, params.lambda_, rng) for _ in range(n_runs)]


def simulate_diff(gamma, noise_set):
    """Differentiable-in-gamma FWHM samples over a fixed noise set."""
    out = []
    for u, b, n in noise_set:
        if n + len(b) < 3:
            out.append(2.0 * gamma)            # differentiable proxy
        else:
            out.append(DifferentiableRun.apply(gamma, u, b))
    return torch.stack(out)


def simulate_target(gamma, n_runs, params, rng):
    """Detached FWHM samples (forward only) — used to build the real-data target."""
    vals = [run_fwhm(torch.tensor(float(gamma)), u, b)
            for u, b, n in (draw_fixed_noise(params.nbar, params.sigma, params.lambda_, rng) for _ in range(n_runs))]
    return torch.tensor(vals)

## Step 6: Loss Function — Biased MMD²

Both sides are sets of FWHM samples, so we compare them directly with the biased
MMD². We apply it in `log10` space (FWHM spans orders of magnitude → a single
kernel bandwidth is only meaningful on a log scale).

In [6]:
def gaussian_gram(a, b, bandwidth):
    """Gaussian kernel (Gram) matrix between two 1-D sample sets."""
    d2 = (a[:, None] - b[None, :]) ** 2
    return torch.exp(-d2 / (2.0 * bandwidth ** 2))


def median_bandwidth(x, y):
    """Median-heuristic bandwidth from pooled samples (detached = constant)."""
    z = torch.cat([x, y]).detach()
    d2 = (z[:, None] - z[None, :]) ** 2
    iu = torch.triu_indices(len(z), len(z), offset=1)
    return torch.median(d2[iu[0], iu[1]]).sqrt().clamp_min(1e-12)


def mmd2_biased(x, y, bandwidth=None):
    """Biased MMD^2 between sample sets x (sim) and y (real), Gaussian kernel."""
    if bandwidth is None:
        bandwidth = median_bandwidth(x, y)
    Kxx = gaussian_gram(x, x, bandwidth)
    Kyy = gaussian_gram(y, y, bandwidth)
    Kxy = gaussian_gram(x, y, bandwidth)
    return Kxx.mean() + Kyy.mean() - 2.0 * Kxy.mean()


def log_fwhm(x):
    """log10 of FWHM samples, floored to stay finite on degenerate fits."""
    return torch.log10(x.clamp_min(1e-3))


# Smoke test: matching params -> much lower MMD^2 than mismatched.
_p = MCParams(nbar=50.0)
_real = log_fwhm(simulate_target(20.0, 400, _p, np.random.default_rng(123)))
_ns = draw_noise_set(400, _p, np.random.default_rng(456))
with torch.no_grad():
    _good = mmd2_biased(log_fwhm(simulate_diff(torch.tensor(20.0), _ns)), _real)
    _bad = mmd2_biased(log_fwhm(simulate_diff(torch.tensor(10.0), _ns)), _real)
print(f"MMD^2 gamma=20 (match): {_good:.6f}")
print(f"MMD^2 gamma=10 (wrong): {_bad:.6f}  (should be larger)")

MMD^2 gamma=20 (match): 0.001238
MMD^2 gamma=10 (wrong): 0.498196  (should be larger)


## Step 7: Validation — Per-run Gradient vs Finite Difference

The key correctness gate: the implicit-diff gradient `dFWHM/dgamma` for a single
run must match a central finite difference of that run's FWHM (same fixed noise).
We expect a small median relative error; large errors only on runs whose true
gradient is ~0 (degenerate fits).

In [7]:
rng = np.random.default_rng(0)
params = MCParams(nbar=50.0)
gamma0 = 20.0
rels = []
for _ in range(60):
    u, b, n = draw_fixed_noise(params.nbar, params.sigma, params.lambda_, rng)
    if n + len(b) < 5:
        continue
    g = torch.tensor(gamma0, requires_grad=True)
    fwhm = DifferentiableRun.apply(g, u, b)
    fwhm.backward()
    d_ift = g.grad.item()
    h = 1e-2
    d_fd = (run_fwhm(torch.tensor(gamma0 + h), u, b)
            - run_fwhm(torch.tensor(gamma0 - h), u, b)) / (2 * h)
    if abs(d_fd) > 1e-2:
        rels.append(abs(d_ift - d_fd) / abs(d_fd))
rels = np.array(rels)
print(f"per-run gradient check over {len(rels)} runs:")
print(f"  median rel err = {np.median(rels):.2%}")
print(f"  90th pct       = {np.quantile(rels, 0.9):.2%}")

per-run gradient check over 60 runs:
  median rel err = 6.24%
  90th pct       = 96.75%


## Step 8: End-to-end Recovery of gamma

The real test: generate "real" data at a known `gamma_true`, start from a wrong
`gamma`, and recover it by gradient descent on the log-space MMD² — gradients
flowing through the fit via implicit differentiation. `nbar` is held fixed.

For speed we use a few hundred runs per step during optimization (full 2000 is only
needed for final reporting). Fixed noise is resampled each step but held fixed
within a step (required for the implicit-diff gradient).

In [8]:
GAMMA_TRUE = 20.0
params = MCParams(nbar=50.0)
N_RUNS = 200          # a few hundred runs is enough gradient signal; bump for final eval

real = log_fwhm(simulate_target(GAMMA_TRUE, 400, params, np.random.default_rng(2024)))

gamma = torch.tensor(10.0, requires_grad=True)   # deliberately wrong start
opt = torch.optim.Adam([gamma], lr=0.5)
rng_step = np.random.default_rng(0)

history = []
for step in range(70):
    noise = draw_noise_set(N_RUNS, params, rng_step)
    opt.zero_grad()
    loss = mmd2_biased(log_fwhm(simulate_diff(gamma, noise)), real)
    loss.backward()
    opt.step()
    with torch.no_grad():
        gamma.clamp_(min=0.5, max=70.0)
    history.append((step, gamma.item(), loss.item()))
    if step % 10 == 0 or step == 69:
        print(f"step {step:3d}  gamma={gamma.item():6.3f}  loss={loss.item():.6f}")

recovered = np.mean([h[1] for h in history[-15:]])
print(f"\nrecovered gamma = {recovered:.3f}  (true {GAMMA_TRUE})")

step   0  gamma=10.500  loss=0.601759


step  10  gamma=15.293  loss=0.149137


step  20  gamma=19.035  loss=0.015903


In [ ]:
steps = [h[0] for h in history]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(steps, [h[1] for h in history])
ax[0].axhline(GAMMA_TRUE, color="k", ls="--", lw=1, label="gamma_true")
ax[0].set(xlabel="step", ylabel="gamma", title="Recovery of gamma")
ax[0].legend()
ax[1].plot(steps, [h[2] for h in history])
ax[1].set(xlabel="step", ylabel="MMD$^2$ (log-space)", title="Loss", yscale="log")
plt.tight_layout()
plt.show()